# BreastDCEDL on Colab A100 — End-to-End pCR Prediction

Reproduces Fridman et al. 2025 (paper Table 2). **Top-down runnable** —
Runtime → Run all on A100 (40 GB).

**Prerequisites** (one-time setup, then re-runs use the cache):
- Drive folder `MyDrive/breastdcedl/data/` contains:
  - `BreastDCEDL_ISPY1_min_crop.tar.gz` (~1.3 GB)
  - `BreastDCEDL_ISPY2_min_crop.tar.gz` (~13.6 GB)
  - `BreastDCEDL_DUKE_min_crop.tar.gz` (~8.6 GB)
  - `BreastDCEDL_models.tar.gz` (~320 MB) — paper's pretrained .pth
  - `BreastDCEDL_metadata_min_crop.csv` (~340 KB)

Pipeline: extract → ViT-Base + paper's pretrained → 30-epoch two-phase
fine-tune (head-only warmup → LLRD) → patient-level eval + 4-view TTA →
subtype + cohort breakdown.


## 1. Runtime check (A100, bf16, fix Colab triton if needed)


In [ ]:
# Colab sometimes ships a triton whose API has drifted from the installed torch
# (`AttributeError: module 'triton.language' has no attribute 'core'`). Upgrade
# before any torch import so the smoke cell doesn't crash later.
!pip install -q -U triton 2>&1 | tail -2

import torch, platform
print(f"python   : {platform.python_version()}")
print(f"torch    : {torch.__version__}")
print(f"cuda     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    if 'A100' not in torch.cuda.get_device_name(0):
        print("WARNING: not an A100 — bf16 may fall back to fp16 (slower).")
else:
    raise SystemExit("No CUDA device. Runtime → Change runtime type → A100.")


## 2. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT  = '/content/drive/MyDrive/breastdcedl'
DATA_DIR    = f'{DRIVE_ROOT}/data'         # archives live here, persistent
CKPT_DIR    = f'{DRIVE_ROOT}/checkpoints'  # checkpoints persist across runtimes
RESULTS_DIR = f'{DRIVE_ROOT}/results'

assert os.path.isdir(DATA_DIR), (
    f'{DATA_DIR} not found. Place archives + CSV under MyDrive/breastdcedl/data/ '
    'before running this notebook.'
)
os.makedirs(CKPT_DIR,    exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')
print(f'Data dir  : {DATA_DIR}')


## 3. Clone repo & install dependencies


In [ ]:
import os
os.chdir('/content')   # in case the prior CWD was wiped by a runtime restart

REPO_URL = 'https://github.com/javaapplesauce/BreastDCEDL-4-GILM.git'
REPO_DIR = '/content/breastdcedl'

# Shallow clone — repo history contains ~1.2 GB of committed sample files.
!rm -rf {REPO_DIR}
!git clone --depth=1 --single-branch {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!bash colab/bootstrap.sh {REPO_DIR}


In [ ]:
import sys, os

REPO_DIR = '/content/breastdcedl'
assert os.path.isdir(REPO_DIR), f'{REPO_DIR} missing — re-run the clone cell above.'
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Guarantee namespace packages have __init__.py.
for sub in ('src', 'src/data', 'src/models', 'src/training', 'src/evaluation'):
    init = os.path.join(REPO_DIR, sub, '__init__.py')
    if os.path.isdir(os.path.dirname(init)) and not os.path.isfile(init):
        open(init, 'a').close()

print('cwd :', os.getcwd())
print('path:', REPO_DIR, 'on sys.path')


## 4. Extract archives from Drive cache → local disk

Archives live persistently on Drive; extracted trees go to `/content/data/`
because Drive FUSE is ~100× slower for many-small-files. Re-runs of this cell
skip cohorts that are already extracted on local disk.


In [ ]:
import os, shutil, tarfile, time

DRIVE_CACHE = DATA_DIR              # /content/drive/MyDrive/breastdcedl/data
LOCAL_DATA  = '/content/data'
os.makedirs(LOCAL_DATA, exist_ok=True)

# (drive filename, expected extracted folder)
ARCHIVES = [
    ('BreastDCEDL_ISPY1_min_crop.tar.gz', 'BreastDCEDL_ISPY1_min_crop'),
    ('BreastDCEDL_ISPY2_min_crop.tar.gz', 'BreastDCEDL_ISPY2_min_crop'),
    ('BreastDCEDL_DUKE_min_crop.tar.gz',  'BreastDCEDL_DUKE_min_crop'),
    ('BreastDCEDL_models.tar.gz',         'BreastDCEDL_models'),
]
NON_ARCHIVES = ['BreastDCEDL_metadata_min_crop.csv']

print('=== Drive cache contents ===')
for f in sorted(os.listdir(DRIVE_CACHE)):
    p = os.path.join(DRIVE_CACHE, f)
    if os.path.isfile(p):
        print(f'  {f:55s} {os.path.getsize(p)/1e6:>9.1f} MB')

print('\n=== Extracting archives ===')
for fname, ext_dir in ARCHIVES:
    src = os.path.join(DRIVE_CACHE, fname)
    dst = os.path.join(LOCAL_DATA, ext_dir)
    if os.path.isdir(dst) and any(os.scandir(dst)):
        print(f'  [skip]     {ext_dir}/  already extracted')
        continue
    if not os.path.isfile(src):
        raise FileNotFoundError(f'Drive missing required archive: {src}')
    if os.path.getsize(src) < 1_000_000:
        raise ValueError(f'{src} is suspiciously small ({os.path.getsize(src)} bytes) — re-upload to Drive.')
    print(f'  [extract]  {fname}  → {LOCAL_DATA}')
    t0 = time.time()
    with tarfile.open(src, 'r:gz') as t:
        t.extractall(LOCAL_DATA)
    print(f'    done in {time.time()-t0:.0f}s')

print('\n=== Copying non-archives ===')
for fname in NON_ARCHIVES:
    src = os.path.join(DRIVE_CACHE, fname)
    dst = os.path.join(LOCAL_DATA, fname)
    if os.path.isfile(dst) and os.path.getsize(dst) > 1000:
        print(f'  [ok]       {fname}  ({os.path.getsize(dst)/1e3:.1f} KB)')
        continue
    if not os.path.isfile(src) or os.path.getsize(src) < 1000:
        raise FileNotFoundError(f'Drive missing or corrupt: {src}')
    shutil.copy(src, dst)
    print(f'  [copy]     {fname}  → local ({os.path.getsize(dst)/1e3:.1f} KB)')

DATA_DIR = LOCAL_DATA  # downstream cells read from local
print(f'\nDATA_DIR is now: {DATA_DIR}')
!du -sh {DATA_DIR}/* 2>/dev/null | sort -h | tail -8

meta = os.path.join(LOCAL_DATA, 'BreastDCEDL_metadata_min_crop.csv')
assert os.path.isfile(meta) and os.path.getsize(meta) > 1000, (
    f'metadata CSV invalid: exists={os.path.isfile(meta)} '
    f'size={os.path.getsize(meta) if os.path.isfile(meta) else "n/a"}'
)
print(f'[verify] metadata CSV: {os.path.getsize(meta)/1e3:.1f} KB OK')


## 5. Discover NIfTI/mask dirs + pretrained weights


In [ ]:
from pathlib import Path

cohort_roots = {
    'spy1': f'{DATA_DIR}/BreastDCEDL_ISPY1_min_crop',
    'spy2': f'{DATA_DIR}/BreastDCEDL_ISPY2_min_crop',
    'duke': f'{DATA_DIR}/BreastDCEDL_DUKE_min_crop',
}

nifti_paths, mask_paths = {}, {}
for cohort, root in cohort_roots.items():
    if not os.path.isdir(root):
        print(f'  [skip] {cohort}: {root} not present')
        continue
    mdir = next(
        (str(p) for p in sorted(Path(root).rglob('*'))
         if p.is_dir() and 'mask' in p.name.lower()),
        None,
    )
    ndir = None
    for p in sorted(Path(root).rglob('*.nii.gz')):
        parent = str(p.parent)
        if mdir and parent.startswith(mdir):
            continue
        ndir = parent; break

    if ndir: nifti_paths[cohort] = ndir
    if mdir: mask_paths[cohort]  = mdir
    n_nifti = len(list(Path(ndir).glob('*.nii.gz'))) if ndir else 0
    n_mask  = len(list(Path(mdir).glob('*.nii.gz'))) if mdir else 0
    print(f'  {cohort:5s} dce={ndir}  ({n_nifti} files)')
    print(f'  {cohort:5s} msk={mdir}  ({n_mask} files)')

pth_files = sorted(Path(DATA_DIR).rglob('*.pth'))
print(f'\nPretrained .pth files:')
for p in pth_files:
    print(f'  {p.relative_to(DATA_DIR)}  ({p.stat().st_size/1e6:.0f} MB)')
PRETRAINED = str(pth_files[0]) if pth_files else None
print(f'\nSelected pretrained: {PRETRAINED}')

METADATA_CSV = f'{DATA_DIR}/BreastDCEDL_metadata_min_crop.csv'
assert PRETRAINED is not None, 'no .pth files found in DATA_DIR — extract models archive'
assert os.path.isfile(METADATA_CSV), f'metadata missing: {METADATA_CSV}'


## 6. Configure training (writes `configs/run.yaml`)


In [ ]:
import yaml

with open('configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['data']['nifti'] = nifti_paths
cfg['data']['masks'] = mask_paths
cfg['data']['combined_metadata'] = METADATA_CSV
cfg['data']['crop_size'] = 224
cfg['data']['n_slices'] = 8
cfg['data']['label_col'] = 'pCR'

# ViT-Base matches the layer naming of the paper's pretrained .pth so the
# backbone+classifier transfer cleanly. DINOv2-Base would drop ~225 keys.
cfg['model']['backbone'] = 'google/vit-base-patch16-224-in21k'
cfg['model']['pretrained_weights'] = PRETRAINED
cfg['model']['dropout'] = 0.3

# A100 40 GB: batch 48 × accum 1 fits comfortably at 224px / ViT-Base / bf16.
cfg['training']['batch_size'] = 48
cfg['training']['accum_steps'] = 1
cfg['training']['num_epochs'] = 30
cfg['training']['freeze_epochs'] = 5
cfg['training']['backbone_lr'] = 5e-6
cfg['training']['head_lr'] = 5e-4
cfg['training']['llrd'] = 0.85
cfg['training']['patience'] = 10
cfg['training']['num_workers'] = 8
cfg['training']['amp_dtype'] = 'bf16'
cfg['training']['loss'] = 'focal'

cfg['wandb'] = {'enabled': False, 'project': 'breastdcedl-vit', 'entity': None,
                'tags': ['a100', 'vit-base', 'pcr', 'mincrop']}
cfg['checkpoint_dir'] = CKPT_DIR

with open('configs/run.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)
print('wrote configs/run.yaml')


## 7. Verify data loading (visualize 4 RGB-fused patients)


In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.data.preprocessing import setup_paths, load_acquisitions, fuse_rgb_slice, select_timepoints, _cohort_from_pid
from src.data.splits import load_and_split

setup_paths(nifti_dirs=nifti_paths, mask_dirs=mask_paths)

train_df, val_df, test_df = load_and_split(METADATA_CSV, label_col='pCR', seed=42)

# Filter to cohorts we actually have on disk.
avail = set(nifti_paths.keys())
if 'dataset' in train_df.columns:
    train_df = train_df[train_df['dataset'].isin(avail)].reset_index(drop=True)
    val_df   = val_df  [val_df  ['dataset'].isin(avail)].reset_index(drop=True)
    test_df  = test_df [test_df ['dataset'].isin(avail)].reset_index(drop=True)

print(f'train={len(train_df)}  val={len(val_df)}  test={len(test_df)}')
print(f'pCR rates: train={train_df.pCR.mean():.1%}  val={val_df.pCR.mean():.1%}  test={test_df.pCR.mean():.1%}')

# Quick visual sanity check
sample = train_df.sample(min(4, len(train_df)), random_state=1)
fig, axes = plt.subplots(1, len(sample), figsize=(4*len(sample), 4))
if len(sample) == 1: axes = [axes]
for ax, (_, row) in zip(axes, sample.iterrows()):
    pid = row['pid']
    acqs = load_acquisitions(pid)
    if acqs is None or len(acqs) < 2:
        ax.set_title(f'{pid}\n(no data)'); ax.axis('off'); continue
    cohort = _cohort_from_pid(pid)
    ip = row.get('pre');        ip = None if pd.isna(ip) else ip
    ie = row.get('post_early'); ie = None if pd.isna(ie) else ie
    il = row.get('post_late');  il = None if pd.isna(il) else il
    pre, early, late = select_timepoints(acqs, cohort, idx_pre=ip, idx_early=ie, idx_late=il)
    z = pre.shape[2] // 2
    rgb = fuse_rgb_slice(pre[:, :, z], early[:, :, z], late[:, :, z])
    ax.imshow(rgb); ax.axis('off')
    ax.set_title(f'{pid[:18]}...\npCR={int(row.pCR)}  {cohort}')
plt.suptitle('RGB fusion (R=pre, G=early-post, B=late-post)')
plt.tight_layout(); plt.show()


## 8. (Optional) Resume from disconnect

Auto-detects the latest checkpoint on Drive. If you changed the backbone or
training config, the trainer will detect the mismatch and train from scratch
without crashing — you can manually move stale checkpoints out of the way:
`!mkdir -p {CKPT_DIR}/old && mv {CKPT_DIR}/epoch_*.pth {CKPT_DIR}/best.pth {CKPT_DIR}/old/ 2>/dev/null`


In [ ]:
import os, glob

candidates = sorted(glob.glob(f'{CKPT_DIR}/epoch_*.pth'))
if candidates:
    latest = candidates[-1]
    cfg['resume'] = latest
    print(f"[resume] cfg['resume'] = {latest}")
    print(f"[resume] num_epochs    = {cfg['training']['num_epochs']}")
    print(f"[resume] freeze_epochs = {cfg['training']['freeze_epochs']}")
else:
    print(f'[resume] no epoch_*.pth in {CKPT_DIR} — training from scratch')


## 9. Train (30 epochs: 5 head-only warmup + 25 LLRD fine-tune)

~45-60 min on A100. Checkpoints save to Drive after every epoch as full state
(model + optimizer + scheduler + epoch + history) so a disconnect resumes
losslessly.


In [ ]:
from scripts.train import train_from_config

best_auc = train_from_config(cfg)
print(f'\nBest validation AUC: {best_auc:.4f}')
print(f'Checkpoints in {CKPT_DIR}')


## 10. Training curves


In [ ]:
import json
with open(f'{CKPT_DIR}/history.json') as f:
    history = json.load(f)

epochs = [e['epoch'] for e in history]
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(epochs, [e['train_loss'] for e in history], label='train')
ax[0].plot(epochs, [e['val_loss']   for e in history], label='val')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('loss'); ax[0].legend(); ax[0].set_title('Loss')

ax[1].plot(epochs, [e['train_acc'] for e in history], label='train')
ax[1].plot(epochs, [e['accuracy']  for e in history], label='val (patient)')
ax[1].set_xlabel('epoch'); ax[1].set_ylabel('accuracy'); ax[1].legend(); ax[1].set_title('Accuracy')

ax[2].plot(epochs, [e['auc'] for e in history], color='tab:green', label='val AUC')
ax[2].axhline(0.72, linestyle='--', color='grey', alpha=0.7, label='paper overall (0.72)')
ax[2].axhline(0.94, linestyle='--', color='red',  alpha=0.7, label='paper HR+/HER2− (0.94)')
ax[2].set_xlabel('epoch'); ax[2].set_ylabel('AUC'); ax[2].legend(); ax[2].set_title('Patient-level AUC')

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/training_curves.png', dpi=150)
plt.show()


## 11. Evaluate on held-out test set (patient-level + 4-view TTA)


In [ ]:
from torch.utils.data import DataLoader
from src.data.dataset import BreastDCEDataset, build_transforms
from src.models.vit import BreastDCEViT
from src.evaluation.metrics import compute_metrics, evaluate_by_subtype, predict_with_tta

device = torch.device('cuda')
n_slices  = cfg['data']['n_slices']
crop_size = cfg['data']['crop_size']
label_col = cfg['data']['label_col']

mcfg = cfg['model']
use_clinical  = mcfg.get('use_clinical', False)
clinical_cols = mcfg.get('clinical_features') if use_clinical else None
clinical_dim  = mcfg.get('clinical_dim', 0) if use_clinical else 0

test_ds = BreastDCEDataset(
    test_df, label_col=label_col,
    crop_size=crop_size, n_slices=n_slices,
    transform=build_transforms({}, is_train=False),
    clinical_cols=clinical_cols,
)
# CRITICAL: shuffle=False — patient_level pooling assumes contiguous patient ordering.
test_loader = DataLoader(
    test_ds, batch_size=cfg['training']['batch_size'],
    shuffle=False, num_workers=cfg['training']['num_workers'], pin_memory=True,
    persistent_workers=cfg['training']['num_workers'] > 0,
)

eval_model = BreastDCEViT(
    backbone=mcfg['backbone'], num_classes=mcfg['num_classes'], dropout=0.0,
    use_clinical=use_clinical, clinical_dim=clinical_dim,
).to(device)

# Dual-format checkpoint loader: full-state dict OR bare state_dict.
best_path = f"{CKPT_DIR}/best.pth"
blob = torch.load(best_path, map_location=device, weights_only=False)
state = blob['model'] if isinstance(blob, dict) and 'model' in blob else blob
eval_model.load_state_dict(state)
eval_model.eval()
print(f'loaded {best_path}  use_clinical={use_clinical}'
      + (f"  (epoch {blob['epoch']}, AUC {blob.get('best_auc', 0):.4f})"
         if isinstance(blob, dict) and 'epoch' in blob else '  (legacy bare state_dict)'))

# 4-view TTA (identity + h-flip + v-flip + hv-flip), patient-mean pooling.
y_true, y_prob, y_pred = predict_with_tta(
    eval_model, test_loader, device,
    n_slices=n_slices, use_clinical=use_clinical, tta=True,
)
print(f'test set: {len(y_true)} patients')
overall = compute_metrics(y_true, y_prob, y_pred)
print('\n=== Overall Test Set ===')
for k, v in overall.items():
    print(f'  {k:13s}: {v:.4f}')

print('\n=== Paper baseline (Table 2, overall test) ===')
print('  AUC            : 0.72')
print('  Accuracy       : 0.75')
print('  Sensitivity    : 0.27')
print('  Specificity    : 0.95')


## 12. Subtype + per-cohort breakdown

Paper's headline 0.94 AUC is on **I-SPY2 only + HR+/HER2− + clinical features**
(Table 2 bottom block). The cell below adds an I-SPY2-only sub-analysis so we
can compare apples-to-apples even when use_clinical=False.


In [ ]:
subtypes = cfg.get('evaluation', {}).get('subtypes', {
    'HR+/HER2-': {'HRposHER2neg': 1},
    'HER2+':     {'HER2pos': 1},
    'TripleNeg': {'TripleNeg': 1},
})
sub_df = evaluate_by_subtype(test_df.reset_index(drop=True), y_prob, y_pred, label_col=label_col, subtypes=subtypes)
print('\n=== Subtype breakdown (test set, all cohorts) ===')
print(sub_df.to_string(index=False, float_format='%.3f'))
sub_df.to_csv(f'{RESULTS_DIR}/subtype_results.csv', index=False)

test_reset = test_df.reset_index(drop=True)
if 'pid' in test_reset.columns:
    rows = []
    for name, pat in [('Duke','Breast_MRI'),('I-SPY1','ISPY1'),('I-SPY2','ISPY2'),('ACRIN-6698','ACRIN-6698')]:
        m = test_reset['pid'].str.contains(pat, na=False).values
        if m.sum() < 3: continue
        rows.append({'Cohort': name, 'N': int(m.sum()), **compute_metrics(y_true[m], y_prob[m], y_pred[m])})
    cohort_df = pd.DataFrame(rows)
    if len(cohort_df):
        print('\n=== Per-cohort breakdown (test set) ===')
        print(cohort_df.to_string(index=False, float_format='%.3f'))
        cohort_df.to_csv(f'{RESULTS_DIR}/cohort_results.csv', index=False)

# I-SPY2-only sub-analysis (paper Table 2 bottom block).
if 'dataset' in test_reset.columns:
    ispy2_mask = (test_reset['dataset'] == 'spy2').values
    if ispy2_mask.sum() >= 5:
        ispy2_df = test_reset[ispy2_mask].reset_index(drop=True)
        ispy2_sub = evaluate_by_subtype(
            ispy2_df, y_prob[ispy2_mask], y_pred[ispy2_mask],
            label_col=label_col, subtypes=subtypes,
        )
        print('\n=== I-SPY2 only (paper Table 2 bottom block) ===')
        print(ispy2_sub.to_string(index=False, float_format='%.3f'))
        ispy2_sub.to_csv(f'{RESULTS_DIR}/ispy2_subtype_results.csv', index=False)
        print(f"\nNote: paper's 0.94 HR+/HER2- AUC also requires use_clinical=True;")
        print(f"      current run has use_clinical={cfg['model'].get('use_clinical', False)}.")


## 13. ROC + confusion matrix


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix as cm_fn
import seaborn as sns

fpr, tpr, _ = roc_curve(y_true, y_prob)
auc_val = roc_auc_score(y_true, y_prob)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(fpr, tpr, lw=2, label=f'AUC = {auc_val:.3f}')
ax[0].plot([0, 1], [0, 1], '--', color='grey')
ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR'); ax[0].legend()
ax[0].set_title('Test ROC'); ax[0].set_aspect('equal')

cm = cm_fn(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-pCR','pCR'], yticklabels=['Non-pCR','pCR'], ax=ax[1])
ax[1].set_xlabel('Predicted'); ax[1].set_ylabel('True'); ax[1].set_title('Confusion Matrix')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/roc_cm.png', dpi=150); plt.show()


## 14. Save predictions + artifacts to Drive


In [ ]:
pred_df = test_df.reset_index(drop=True)[['pid']].copy() if 'pid' in test_df.columns else pd.DataFrame(index=range(len(y_true)))
pred_df['y_true'] = y_true
pred_df['y_prob'] = y_prob
pred_df['y_pred'] = y_pred
pred_df.to_csv(f'{RESULTS_DIR}/predictions.csv', index=False)

!cp -f {CKPT_DIR}/history.json {RESULTS_DIR}/history.json 2>/dev/null || true
!cp -f {CKPT_DIR}/best.pth     {RESULTS_DIR}/best.pth     2>/dev/null || true

print(f'Results saved to {RESULTS_DIR}:')
!ls -la {RESULTS_DIR}
